In [ ]:
import os
import csv
import shutil
import pandas as pd
from datetime import datetime
from contextlib import contextmanager
from PIL import Image
import win32wnet
import win32netcon

# Suppress paramiko logs
import logging
logging.getLogger("paramiko").setLevel(logging.WARNING)

# Load configuration
import json
with open("config.json") as f:
    config = json.load(f)

SFTP_ROOT = config["backup_folder_root"]
PROCESSED_ROOT = config["processed_images_root"]
LOG_ROOT = config["logs_root"]
NETWORK_CONFIG = config["network"]

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

@contextmanager
def network_access(path, username, password, domain=""):
    net_resource = win32wnet.NETRESOURCE()
    net_resource.lpRemoteName = os.path.dirname(path)
    net_resource.dwType = win32netcon.RESOURCETYPE_DISK
    try:
        win32wnet.WNetAddConnection2(net_resource, password, f"{domain}\\{username}" if domain else username, 0)
        yield
    finally:
        win32wnet.WNetCancelConnection2(net_resource.lpRemoteName, 0, 1)

def log(msg, logfile):
    with open(logfile, "a", encoding="utf-8") as f:
        f.write(f"{datetime.now().isoformat()} - {msg}\n")

# def compress_image(path, error_log):
#     try:
#         orig_size = os.path.getsize(path)
#         if orig_size < 300_000:
#             return False, orig_size, orig_size

#         img = Image.open(path)
#         img = img.convert("RGB")
#         img.save(path, optimize=True, quality=60)

#         comp_size = os.path.getsize(path)
#         return True, orig_size, comp_size
#     except Exception as e:
#         log(f"ERROR: Failed to compress {path} | {e}", error_log)
#         return False, 0, 0


def compress_image(path, error_log):
    try:
        orig_size = os.path.getsize(path)
        if orig_size < 300_000:
            return False, orig_size, orig_size

        ext = os.path.splitext(path)[1].lower()
        img = Image.open(path).convert("RGB")

        # Remove metadata by re-creating image
        data = list(img.getdata())
        clean_img = Image.new(img.mode, img.size)
        clean_img.putdata(data)

        # Save with compression settings based on file type
        if ext in [".jpg", ".jpeg"]:
            clean_img.save(path, "JPEG", quality=60, optimize=True, progressive=True)
        elif ext == ".png":
            clean_img.save(path, "PNG", optimize=True)

        comp_size = os.path.getsize(path)
        return True, orig_size, comp_size

    except Exception as e:
        log(f"ERROR: Failed to compress {path} | {e}", error_log)
        return False, 0, 0
        

# def copy_to_backup(path, backup_folder, error_log):
#     try:
#         os.makedirs(backup_folder, exist_ok=True)
#         shutil.copy2(path, os.path.join(backup_folder, os.path.basename(path)))
#         return True
#     except Exception as e:
#         log(f"ERROR: Failed to backup {path} | {e}", error_log)
#         return False


def sftp_backup_file(local_file, folder_name, file_name, sftp):
    remote_folder = os.path.join(BACKUP_ROOT, folder_name, RUN_TIMESTAMP).replace("\\", "/")
    remote_file = os.path.join(remote_folder, file_name).replace("\\", "/")

    try:


        try:
            sftp.chdir(remote_folder)
        except IOError:
            parts = remote_folder.strip("/").split("/")
            current_path = ""
            for part in parts:
                current_path = f"{current_path}/{part}"
                try:
                    sftp.chdir(current_path)
                except IOError:
                    sftp.mkdir(current_path)
                    sftp.chdir(current_path)

        sftp.put(local_file, remote_file)
        sftp.close()
        ssh.close()
        return True, None

    except Exception as e:
        return False, str(e)
        

def get_all_processed_files(folder_log_dir):
    all_processed_files = set()
    if not os.path.exists(folder_log_dir):
        return all_processed_files
    for fname in os.listdir(folder_log_dir):
        print(f"existing_files: {folder_log_dir} ")
        if fname.startswith("processed_") and fname.endswith(".csv"):
            fpath = os.path.join(folder_log_dir, fname)
            try:
                df = pd.read_csv(fpath)
                all_processed_files.update(df['image-file-name'].dropna().tolist())
            except Exception:
                pass
    return all_processed_files

def write_to_timestamped_processed(folder_name, row):
    folder_log_dir = os.path.join(LOG_ROOT, folder_name)
    os.makedirs(folder_log_dir, exist_ok=True)
    timestamped_file = os.path.join(folder_log_dir, f"processed_{folder_name}_{RUN_TIMESTAMP}.csv")
    file_exists = os.path.exists(timestamped_file)
    with open(timestamped_file, "a", newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["image-file-name", "original-size", "isBackedUp", "isCompressed",
                             "compressed-size", "file-original-created-datetime", "modified-datetime", "isProcessed"])
        writer.writerow(row)

def process_folder(path, folder_name):
    folder_log_dir = os.path.join(LOG_ROOT, folder_name)
    success_log = os.path.join(folder_log_dir, "success.log")
    failure_log = os.path.join(folder_log_dir, "failure.log")
    error_log = os.path.join(folder_log_dir, "error.log")

    os.makedirs(folder_log_dir, exist_ok=True)

    existing_files = get_all_processed_files(folder_log_dir)
    dated_backup_folder = os.path.join(SFTP_ROOT, folder_name, RUN_TIMESTAMP)
    #print(f"existing_files: {existing_files} ")

    try:
        ssh = paramiko.SSHClient()
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        ssh.connect(SFTP_CONFIG["host"], SFTP_CONFIG["port"], SFTP_CONFIG["username"], SFTP_CONFIG["password"])
        sftp = ssh.open_sftp()
    
        files = []
        for root, _, filenames in os.walk(path):
            for name in filenames:
                if name.lower().endswith(('jpg', 'jpeg', 'png')) and name not in existing_files:
                    fpath = os.path.join(root, name)
                    ctime = os.path.getctime(fpath)
                    files.append((fpath, ctime))
    
        files.sort(key=lambda x: x[1])
    
        for file_path, ctime in files:
            file_name = os.path.basename(file_path)
            created_str = datetime.fromtimestamp(ctime).strftime('%Y-%m-%d %H:%M:%S')
            mod_time = datetime.fromtimestamp(os.path.getmtime(file_path)).strftime('%Y-%m-%d %H:%M:%S')
    
            #backed_up = copy_to_backup(file_path, dated_backup_folder, error_log)
            orig_file = os.path.join(path, file_name)
            output_file = os.path.join(path, file_name)  # overwrite
            
            backed_up, backup_err = sftp_backup_file(orig_file, folder_name, file_name, sftp)
            
            if not backed_up:
                log(f"FAIL: Backup failed for {file_name}", failure_log)
                row = [file_name, 0, False, False, 0, created_str, mod_time, False]
                write_to_timestamped_processed(folder_name, row)
                continue
    
            compressed, orig_size, comp_size = compress_image(file_path, error_log)
            is_processed = backed_up and (compressed or orig_size < 300_000)
    
            log(f"SUCCESS: {file_name} | Backup: {backed_up} | Compressed: {compressed} | Size: {orig_size//1024}KB -> {comp_size//1024}KB", success_log)
            row = [file_name, orig_size, backed_up, compressed, comp_size, created_str, mod_time, is_processed]
            write_to_timestamped_processed(folder_name, row)

    except Exception as e:
    print(f"[ERROR] Failed to establish or close SFTP connection: {e}")

def process_folders(csv_path):
    df = pd.read_csv(csv_path)
    for _, row in df.iterrows():
        path, folder_name = row["path"], row["folder"]
        with network_access(path, NETWORK_CONFIG['username'], NETWORK_CONFIG['password'], NETWORK_CONFIG.get('domain', "")):
            print(f"path and folder_name: {path} {folder_name}")
            process_folder(path, folder_name)

if __name__ == "__main__":
    process_folders("folders-to-process-ftp.csv")
